In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/netflix_titles.csv')

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:
df.shape

(8807, 12)

Week-2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
print("Null values in each column:")
df.isnull().sum()

duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")

print("--- Unique Ratings ---")
print(df['rating'].unique())

print("\n--- Unique Types ---")
print(df['type'].unique())

Null values in each column:
Total duplicate rows: 0
--- Unique Ratings ---
['PG-13' 'TV-MA' 'PG' 'TV-14' 'TV-PG' 'TV-Y' 'TV-Y7' 'R' 'TV-G' 'G'
 'NC-17' '74 min' '84 min' '66 min' 'NR' nan 'TV-Y7-FV' 'UR']

--- Unique Types ---
['Movie' 'TV Show']


In [ ]:
inconsistent_mask = df['rating'].astype(str).str.contains('min|Season', na=False)
df.loc[inconsistent_mask, 'duration'] = df.loc[inconsistent_mask, 'rating']
df.loc[inconsistent_mask, 'rating'] = 'Unknown'

print("Duplicates and inconsistent categories handled.")

Duplicates and inconsistent categories handled.


In [ ]:
#  Unknown
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')

#  Mode
most_common_country = df['country'].mode()[0]
df['country'] = df['country'].fillna(most_common_country)

most_common_rating = df['rating'].mode()[0]
df['rating'] = df['rating'].fillna(most_common_rating)

df = df.dropna(subset=['date_added', 'duration'])

# Final check
print("Null values remaining after targeted cleaning:")
print(df.isnull().sum())

Null values remaining after targeted cleaning:
show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64


Week-3

In [ ]:
categorical_columns = ['type', 'title', 'director', 'cast', 'country', 'rating', 'listed_in']
for col in categorical_columns:
    df[col] = df[col].astype(str).str.strip().str.title()

columns_with_lists = ['director', 'cast', 'country', 'listed_in']
for col in columns_with_lists:
    df[col] = df[col].str.replace(r'\s*,\s*', ', ', regex=True)

df['genres'] = df['listed_in'].apply(lambda x: [i.strip() for i in x.split(',')])

rating_map = {
    'Tv-Y': 'Kids',   'Tv-Y7': 'Kids',    'G': 'Kids',
    'Tv-G': 'General','Pg': 'General',    'Tv-Pg': 'General',
    'Pg-13': 'Teen',  'Tv-14': 'Teen',
    'R': 'Mature',    'Tv-Ma': 'Mature',  'Nc-17': 'Mature',
    'Nr': 'Unrated',  'Ur': 'Unrated',
}
df['rating_category'] = df['rating'].map(rating_map).fillna('Other')


df['countries_list'] = df['country'].apply(
    lambda x: [c.strip() for c in x.split(',')] if x not in ['Unknown', 'Nan'] else ['Unknown']
)
df['primary_country'] = df['countries_list'].apply(lambda x: x[0])

print("Categorical data normalization and feature engineering completed!")
print(df['rating_category'].value_counts())

Categorical data normalization and feature engineering completed!
rating_category
Mature     4011
Teen       2647
General    1368
Kids        680
Unrated      82
Other         9
Name: count, dtype: int64


In [ ]:
duplicate_count = df.drop(columns=['genres', 'countries_list']).duplicated().sum()
print(duplicate_count)

0


In [ ]:
# date

df['date_added'] = pd.to_datetime(df['date_added'].astype(str).str.strip(), errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month_name() # Gets the actual name like 'September'

df['release_year'] = pd.to_datetime(df['release_year'].astype(str), format='%Y')

# check
print("--- Data Types After Conversion ---")
print(df[['date_added', 'year_added', 'month_added']].dtypes)

print("\n--- Quick Preview of the Dates ---")
print(df[['date_added', 'year_added', 'month_added']].head())



--- Data Types After Conversion ---
date_added     datetime64[ns]
year_added              int32
month_added            object
dtype: object

--- Quick Preview of the Dates ---
  date_added  year_added month_added
0 2021-09-25        2021   September
1 2021-09-24        2021   September
2 2021-09-24        2021   September
3 2021-09-24        2021   September
4 2021-09-24        2021   September


In [ ]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,genres,rating_category,countries_list,primary_country,year_added,month_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020-01-01,Pg-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",[Documentaries],Teen,[United States],United States,2021,September
1,s2,Tv Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021-01-01,Tv-Ma,2 Seasons,"International Tv Shows, Tv Dramas, Tv Mysteries","After crossing paths at a party, a Cape Town t...","[International Tv Shows, Tv Dramas, Tv Mysteries]",Mature,[South Africa],South Africa,2021,September
2,s3,Tv Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",United States,2021-09-24,2021-01-01,Tv-Ma,1 Season,"Crime Tv Shows, International Tv Shows, Tv Act...",To protect his family from a powerful drug lor...,"[Crime Tv Shows, International Tv Shows, Tv Ac...",Mature,[United States],United States,2021,September
3,s4,Tv Show,Jailbirds New Orleans,Unknown,Unknown,United States,2021-09-24,2021-01-01,Tv-Ma,1 Season,"Docuseries, Reality Tv","Feuds, flirtations and toilet talk go down amo...","[Docuseries, Reality Tv]",Mature,[United States],United States,2021,September
4,s5,Tv Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021-01-01,Tv-Ma,2 Seasons,"International Tv Shows, Romantic Tv Shows, Tv ...",In a city of coaching centers known to train I...,"[International Tv Shows, Romantic Tv Shows, Tv...",Mature,[India],India,2021,September
